# 2-Bit Input Compression Pipeline

## Status

Run the cell below to see what TFRecords and weights are already on disk. Use the output to decide which `tfrecords_exist_*` flags to set to `True` in the **Parameters** cell before running the full pipeline.

In [ ]:
!./status.sh

## Clean Previous Run

⚠️ **Only run the cell below if you want to start completely from scratch.** ⚠️

It permanently deletes:
- `smart-pixels/` — all model weights and checkpoints
- `smart-pixels-ml/` — all output parquets
- `best-model/`, `npy/` — cached model copies and numpy arrays
- All `TFR_files/` directories on NAS for the training and test datasets

**Skip this cell** (do not run it) if you want to reuse existing TFRecords and trained weights from a previous run. Set `tfrecords_exist_* = True` in the Parameters cell instead.

## Setup

Edit `sys.path.insert(0, ...)` in the cell below to point to **your** local path to `two_bit_optimization_helpers`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
import random
import numpy as np
import json
import sys
sys.path.insert(0, './two_bit_optimization_helpers') # Change this line to wherever the two_bit_optimization_helpers is located 
from prepare_tfrecords import generate_tfrecords, load_tfrecords
from train import create_model, train, get_best_thresholds, cleanup_models_and_generators, save_performance_parquet
from viz_utils import viz_history
from utils import load_best_model, save_best_model, save_data_as_npy

### Parameters

Set the required variables below before running `Run All Cells`.

In [ ]:
### REQUIRED TO CHANGE ##############################################################################################################################################################################
#dataset_3src_dir='/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3src_16x16_50x12P5_centeredIncidence_parquets'
#dataset_2sc_dir='/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_2sc_16x16_50x12P5_centeredIncidence_parquets'
#
#weights_directory='/data/dajiang/smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/'
#performance_directory_3src='/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_3src_16x16_50x12P5_centeredIncidence/2bit_optimized/'
#performance_directory_2sc='/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_2sc_16x16_50x12P5_centeredIncidence/2bit_optimized/'

# training dataset: 3-source, 16x16, centered incidence (non-ROI)
dataset_3src_dir='/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3src_16x16_50x12P5_centeredIncidence_parquets'
# test dataset: 2-source, 16x16, centered incidence (non-ROI)
dataset_2sc_dir='/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_2sc_16x16_50x12P5_centeredIncidence_parquets'

# training+test dataset: 2-source, uncentered (-9 to 9), ROI geometry (48x192) — VIAS cocotb sim output (~500K rows)
dataset_2su_roi_dir='/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/implementation/testbench/cocotb/release/roi_finder/sim_out/dataset_2su_-9_9_48x192_50x12P5_parquets'
# training+test dataset: 2-source, centered incidence, ROI geometry (48x192) — VIAS cocotb sim output (~1.7M rows)
# use_roi=True selects this dataset
dataset_2s_roi_dir='/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/implementation/testbench/cocotb/release/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets'

# output parquet: evaluated on dataset_3src validation set
performance_directory_3src='smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_3src_16x16_50x12P5_centeredIncidence/2bit_optimized/'
# output parquet: evaluated on dataset_2sc test set
performance_directory_2sc='smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_2sc_16x16_50x12P5_centeredIncidence/2bit_optimized/'
# output parquet: evaluated on dataset_2su_roi test set
performance_directory_2su_roi='smart-pixels-ml/processed_parquets/dataset_2su_roi/test_dataset_2su_-9_9_48x192_50x12P5_roi/2bit_optimized/'
# output parquet: evaluated on dataset_2s_roi test set
performance_directory_2s_roi='smart-pixels-ml/processed_parquets/dataset_2s_roi/test_dataset_2s_48x192_50x12P5_roi/2bit_optimized/'

# For non-quantized models: Conv2D_Max, Conv2D_Full, Conv2D_Slim, Conv1D_Full, Conv1D_Slim, Mlp_Full, Mlp_Slim
# For quantized models: QConv2D_Max, QConv2D_Full, QConv2D_Slim, QConv1D_Full, QConv1D_Slim, QMlp_Full, QMlp_Slim
model_type='Conv2D_Slim'
#####################################################################################################################################################################################################

### OPTIONAL TO CHANGE ##############################################################################################################################################################################
tfrecords_exist_3src=True
tfrecords_exist_2sc=True

tfrecords_exist_2su_roi=False
tfrecords_exist_2s_roi=True

initial_thresholds=[247.8, 668.4, 1662.9]
seed=42

# If you want to SKIP sections of the pipeline, change these flags to True
skip_part_1=False
skip_part_2=False

# Dataset selector:
#   True  → train and test on dataset_2s_roi (48x192 ROI, VIAS, centered incidence, ~1.7M rows)
#   False → train on dataset_3src (16x16, non-ROI), test on dataset_2sc
use_roi=True

# IF SKIPPING PART 1, THESE ARE REQUIRED FOR PART 2:
tfrecords_dir_train=''                                    # Path to TFRecords for train set
tfrecords_dir_val=''                                      # Path to TFRecords for val set
tfrecords_dir_test=''                                     # Path to TFRecords for test dataset
thresholds=[100, 200, 300]                                # CUSTOM THRESHOLDS
labels_scale=[]                                           # CUSTOM labels scale (Max/Full models require 4 elements, Slim models require 3 elements)
#####################################################################################################################################################################################################

### DON'T CHANGE UNLESS YOU KNOW WHAT YOU ARE DOING #################################################################################################################################################
timeslices=2
select_contained=True
train_batch_size=5000
val_batch_size=5000
threshold_offset=80.0
initial_levels=np.array([0.0, 1.0, 2.0, 3.0], dtype=np.float32)   # 2-bit in outputs (same for Part 1 and Part 2)

# epochs1/2=200 is a quick smoke-test (~5 min/part on CPU); loss is still decreasing at epoch 200
# For production results set both to 1000 (several hours on CPU)

epochs1=200 #1000
noise1=[0,80]                                                     # Gaussian noise for Part 1
train_type1='soft_quantize_layer'                                 # soft_quantize_layer training for Part 1
soft_quantize_layer1=True                                         # soft_quantize_layer added to model in Part 1

epochs2=200 # 2000
noise2=-1                                                         # No noise for Part 2
train_type2='2bit_optimized'                                      # 2bit_optimized training for Part 2
soft_quantize_layer2=False                                        # default model architecture in Part 2
#####################################################################################################################################################################################################

# Training and test dataset parameters — controlled by use_roi flag above
if use_roi:
    # ROI: train and test on dataset_2s_roi (VIAS, centered incidence, ~1.7M rows)
    weights_directory               = 'smart-pixels/weights/dataset_2s_roi_weights/'
    performance_directory_train_val = 'smart-pixels-ml/processed_parquets/dataset_2s_roi/test_dataset_2s_roi/2bit_optimized/'
    dataset_train_src               = dataset_2s_roi_dir
    tfrecords_exist_train           = tfrecords_exist_2s_roi
    select_contained_train          = False
    load_roi_train                  = True
    dataset_test_dir                = dataset_2s_roi_dir
    tfrecords_exist_test            = tfrecords_exist_2s_roi
    performance_dir_test            = performance_directory_2s_roi
    load_roi_test                   = True
    select_contained_test           = False
else:
    # Non-ROI: train on dataset_3src, test on dataset_2sc; uses flat numeric pixel columns
    weights_directory               = 'smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/'
    performance_directory_train_val = performance_directory_3src
    dataset_train_src               = dataset_3src_dir
    tfrecords_exist_train           = tfrecords_exist_3src
    select_contained_train          = select_contained
    load_roi_train                  = False
    dataset_test_dir                = dataset_2sc_dir
    tfrecords_exist_test            = tfrecords_exist_2sc
    performance_dir_test            = performance_directory_2sc
    load_roi_test                   = False
    select_contained_test           = select_contained

## Part 1: Optimizing Charge Thresholds

### Set Random Seeds

In [ ]:
if not skip_part_1:
    tf.random.set_seed(seed)
    random.seed(seed)

### Generate & Load TFRecords (with noise)

Converts training dataset parquet files to TFRecord format and loads them into training and validation data generators. The active dataset is `dataset_2s_roi` when `use_roi=True`, or `dataset_3src` when `use_roi=False`.

Gaussian noise (`mean=0e, stdev=80e`) is added to inputs **only in Part 1** to simulate sensor noise and make threshold optimization more robust.

In [ ]:
if not skip_part_1:
    dataset_train_dir, dataset_validation_dir, tfrecords_dir_train, tfrecords_dir_val = generate_tfrecords(
        dataset_dir=dataset_train_src,
        model_type=model_type,
        train_batch_size=train_batch_size,
        val_batch_size=val_batch_size,
        select_contained=select_contained_train,
        timeslices=timeslices,
        tfrecords_exist=tfrecords_exist_train,  # True: load existing TFRecords, False: convert from parquet
        seed=seed,
        load_roi=load_roi_train,
    )
    print(f"TFRecords dirs train: {tfrecords_dir_train}")
    print(f"                 val: {tfrecords_dir_val}")

    training_generator1, validation_generator1 = load_tfrecords(
        tfrecords_dir_train,
        tfrecords_dir_val,
        noise=noise1,  # Gaussian noise [mean, stdev] in electrons; -1 to disable
        seed=seed,
    )
# Outputs: training_generator1, validation_generator1

### Apply Label Scaling to Test TFRecords

Reads the label scaling factors saved during training dataset TFRecord generation and applies them when generating TFRecords for the test dataset. This ensures labels are normalized consistently between the training/validation set and the test set.

In [ ]:
if not skip_part_1:
    with open(f'{tfrecords_dir_val}/metadata.json', 'r') as file:
        data = json.load(file)
    labels_scale = data['labels_scale']  # scaling factors from training dataset, reused for consistent label normalization

    _, tfrecords_dir_test = generate_tfrecords(
        dataset_dir=dataset_test_dir,
        train_batch_size=train_batch_size, 
        val_batch_size=val_batch_size,
        select_contained=select_contained_test,
        timeslices=timeslices, 
        seed=seed, 
        max_workers=1, 
        tfrecords_exist=tfrecords_exist_test,
        model_type=model_type,
        labels_scale=np.array(labels_scale),    # apply same scaling as training dataset
        test_only=True,                         # only generate test TFRecords, no train/val generators
        load_roi=load_roi_test,
    )
    print(f"TFRecords dirs test: {tfrecords_dir_test}")
# Outputs: tfrecords_dir_test

### Train Model (Soft Quantize Layer)

Build and train a model with a **soft quantize layer** appended: a differentiable approximation of the digitization step that allows the 3 charge thresholds to be learned via backpropagation.

The base model architecture is set by `model_type`.

The model weights are discarded after this step: only the learned thresholds matter.

In [ ]:
if not skip_part_1:
    model1 = create_model(
        model_type=model_type,
        timeslices=timeslices,
        soft_quantize_layer=soft_quantize_layer1,  # True: append soft quantize layer for threshold learning
        initial_thresholds=initial_thresholds,      # starting guess for the 3 charge boundaries (in electrons)
        threshold_offset=threshold_offset,          # offset applied to thresholds during initialization
        initial_levels=initial_levels,              # 2-bit output levels [0, 1, 2, 3]
    )
    
    checkpoints_directory1, fingerprint1, history1 = train(
        model=model1,
        model_type=model_type, 
        weights_directory=weights_directory,
        training_generator=training_generator1,
        validation_generator=validation_generator1, 
        timeslices=timeslices,
        train_type=train_type1,  # 'soft_quantize_layer': uses cosine annealing on the quantizer sharpness
        epochs=epochs1,          # up to 1000 epochs; all checkpoints saved (not just best)
        seed=seed, 
    )
    print(f"Epochs completed: {len(history1.history['loss'])}")
    print(f"Best val loss:    {min(history1.history['val_loss']):.4f}")
# Outputs: checkpoints_directory1, fingerprint1, history1

### Extract Thresholds & Digitize Inputs

Scans all checkpoints from **Part 1**, loads the one with the lowest validation loss, and extracts the learned charge thresholds and levels from the soft quantize layer.

These thresholds define how continuous charge values are mapped to 2-bit integers (0–3) in **Part 2**. The model itself is discarded.

In [ ]:
if not skip_part_1:
    thresholds, levels = get_best_thresholds(
        checkpoints=checkpoints_directory1,  # scans all saved checkpoints, picks lowest val_loss
        model_type=model_type,
        timeslices=timeslices,
        initial_thresholds=initial_thresholds,
        threshold_offset=threshold_offset,
        initial_levels=initial_levels,
    )
# Outputs: thresholds, levels

### Cleanup

Deletes **Part 1** model and data generators, clears the TF session, and forces garbage collection to free memory before **Part 2**.

In [ ]:
if not skip_part_1:
    cleanup_models_and_generators([model1, training_generator1, validation_generator1])

## Part 2: Training on Optimized Thresholds

Trains the final model on inputs that have been hard-digitized to 2-bit using the thresholds from **Part 1**. Unlike **Part 1**, there is no soft quantize layer and no noise: the model learns directly on the discrete integer inputs it will see in hardware.

### Load & Digitize TFRecords (2-bit)

Reloads the same training dataset TFRecords used in **Part 1** (`dataset_2s_roi` or `dataset_3src` depending on `use_roi`), but this time **without noise** and with inputs **digitized to 2-bit** using the thresholds learned in **Part 1**.

Each pixel charge value is mapped to a discrete level in `[0, 1, 2, 3]` before being fed to the model — simulating what the actual hardware will do.

In [ ]:
if not skip_part_2:
    training_generator2, validation_generator2 = load_tfrecords(
        tfrecords_dir_train, 
        tfrecords_dir_val,
        noise=noise2,                      # -1: no noise (unlike Part 1)
        digitize=True,                     # apply hard digitization to inputs
        digitize_levels=initial_levels,    # 2-bit output levels [0, 1, 2, 3]
        digitize_thresholds=thresholds,    # thresholds learned in Part 1
        seed=seed,
    )
    print(f"Digitization thresholds: {thresholds}")
# Outputs: training_generator2, validation_generator2

### Reset Random Seeds

Resets TensorFlow and Python random seeds to ensure reproducible weight initialization for the **Part 2** model.

In [ ]:
if not skip_part_2:
    tf.random.set_seed(seed)
    random.seed(seed)

### Train Model (2-bit Optimized)

Builds and trains the **final model** on hard-digitized 2-bit inputs. No soft quantize layer: this is the standard model architecture that will be used for inference.

In [ ]:
if not skip_part_2:
    model2 = create_model(
        model_type=model_type,
        timeslices=timeslices,
        soft_quantize_layer=soft_quantize_layer2,  # False: standard architecture, no quantize layer
    )
    
    checkpoints_directory2, fingerprint2, history2 = train(
        model=model2,
        model_type=model_type, 
        weights_directory=weights_directory,
        training_generator=training_generator2,
        validation_generator=validation_generator2, 
        timeslices=timeslices,
        train_type=train_type2,  # '2bit_optimized': standard training, no annealing scheduler
        epochs=epochs2,          # up to 1000 epochs; all checkpoints saved (not just best)
        seed=seed, 
    )
    print(f"Epochs completed: {len(history2.history['loss'])}")
    print(f"Best val loss:    {min(history2.history['val_loss']):.4f}")
# Outputs: checkpoints_directory2, fingerprint2, history2

### Evaluate on Training Dataset & Save Results

Loads the best **Part 2** checkpoint (lowest validation loss), runs predictions on the validation set, and saves results to a parquet file for downstream analysis.

Output directory is `performance_directory_train_val` — set to `dataset_2s_roi` when `use_roi=True`, or `dataset_3src` when `use_roi=False`.

In [ ]:
if not skip_part_2:
    save_performance_parquet(
        checkpoints=checkpoints_directory2,           # scans all checkpoints, picks lowest val_loss
        output_directory=performance_directory_train_val,  # dataset_2su_roi or dataset_3src depending on use_roi
        test_generator=validation_generator2,         # evaluates on training dataset validation split
        model_type=model_type,
        train_type=train_type2,
        fingerprint=fingerprint2,
        timeslices=2,
        soft_quantize_layer=False,
    )
# Outputs: parquet file saved to performance_directory_train_val

### Cleanup

Deletes **Part 2** model and data generators, clears the TF session, and forces garbage collection to free memory before evaluation on the test dataset.

In [ ]:
if not skip_part_2:
    cleanup_models_and_generators([model2, training_generator2, validation_generator2])

## Evaluation on Test Dataset

Evaluates the **Part 2** model on the test dataset (`dataset_2s_roi` when `use_roi=True`, or `dataset_2sc` when `use_roi=False`). Results are saved as parquet files for downstream analysis.

### Generate & Load Test TFRecords

Generates TFRecords for the test dataset using the label scaling from the training dataset, then loads them as a test generator with inputs hard-digitized using the **Part 1** thresholds.

In [ ]:
# generate (or locate) test TFRecords unconditionally — tfrecords_dir_test is needed for evaluation regardless of skip_part_2
_, tfrecords_dir_test = generate_tfrecords(
    dataset_dir=dataset_test_dir,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    select_contained=select_contained_test,
    timeslices=timeslices,
    seed=seed,
    max_workers=1,
    tfrecords_exist=tfrecords_exist_test,
    model_type=model_type,
    labels_scale=np.array(labels_scale),   # reuse training dataset scaling for consistent normalization
    test_only=True,                        # only generate test TFRecords, no train/val generators
    load_roi=load_roi_test,
)
print(f"TFRecords dirs test: {tfrecords_dir_test}")

# load test generator unconditionally — test_generator is needed by all evaluation cells below,
# even when skip_part_2=True (i.e. Part 2 training was skipped but evaluation still runs)
test_generator = load_tfrecords(
    tfrecords_dir_test=tfrecords_dir_test,
    noise=noise2,                          # -1: no noise
    quantize=False,
    shuffle=True,
    digitize=True,                         # hard-digitize inputs using Part 1 thresholds
    digitize_levels=initial_levels,        # 2-bit output levels [0, 1, 2, 3]
    digitize_thresholds=thresholds,        # thresholds learned in Part 1
    seed=seed,
    test_only=True,                        # only load test generator, no train/val
)
# Outputs: tfrecords_dir_test, test_generator

### Load Model & Best Checkpoint

> **TODO:** This cell and the code cell below are unused.

In [ ]:
#if not skip_part_2:
#    model3 = create_model(
#        model_type=model_type,
#        timeslices=timeslices,
#        soft_quantize_layer=soft_quantize_layer2,
#    )

### Evaluate on Test Dataset & Save Results

Loads the best **Part 2** checkpoint and runs predictions on the test dataset to evaluate generalization to a different detector geometry.

Results are saved as a parquet file to `performance_dir_test` for downstream analysis.

In [ ]:
# no skip_part_2 guard — evaluation should run regardless of whether Part 2 was trained this session;
# requires checkpoints_directory2 and fingerprint2 to be set (either from training or from the parameters cell)
save_performance_parquet(
    checkpoints=checkpoints_directory2,        # scans all checkpoints, picks lowest val_loss
    output_directory=performance_dir_test,
    test_generator=test_generator,             # test dataset (dataset_2sc or dataset_2su_roi)
    model_type=model_type,
    train_type=train_type2,
    fingerprint=fingerprint2,
    timeslices=timeslices,
    soft_quantize_layer=soft_quantize_layer2,
)
# Outputs: parquet file saved to performance_dir_test

## Visualize Training History

Plots training and validation loss over epochs and saves the figure as a PNG. A red marker indicates the best validation loss epoch.

Run the second cell to zoom in on later epochs if early loss dominates the scale.

In [ ]:
if not skip_part_2:  # history2 only exists when Part 2 training ran this session
    # Plot full training history from epoch 1
    # Saves PNG to: {checkpoints_directory2}/history-training_validation_loss_e1.png
    viz_history(history2, title=model_type, prefix=f'{checkpoints_directory2}/history-')

In [ ]:
if not skip_part_2:  # history2 only exists when Part 2 training ran this session
    # Plot training history from epoch 500 onwards — useful to zoom in on later convergence
    # Saves PNG to: {checkpoints_directory2}/history-training_validation_loss_e500.png
    viz_history(history2, title=model_type, start_epoch=500, prefix=f'{checkpoints_directory2}/history-')

## Model Export & Utilities

Optional cells for loading the best model as an object, saving it in multiple formats, exporting test data as numpy arrays, and running predictions directly.

In [ ]:
if not skip_part_2:  # model2 only exists when Part 2 training ran this session
    # Load the best model checkpoint as a model object (needed for cells below)
    # Note: save_performance_parquet already loads the best model internally for evaluation
    model2_best, bestfile = load_best_model(checkpoints_directory2, model2)
    print(f"Best weights: {bestfile}")

In [ ]:
if not skip_part_2:  # model2_best only exists when Part 2 training ran this session
    # Save the best model in multiple formats (h5, keras, json) to the checkpoints directory
    save_best_model(checkpoints_directory2, model2_best)

In [ ]:
# Optional: reload test_generator from TFRecords if not already in memory
# test_generator is already available from the "Generate & Load Test TFRecords" cell above

# Note: calling OptimizedDataGenerator directly is NOT sufficient here —
# it would skip the digitization step and produce continuous inputs instead of 2-bit.
# Use load_tfrecords with the same parameters as the original cell instead:
#
# from OptimizedDataGenerator_v3 import OptimizedDataGenerator
# test_generator = OptimizedDataGenerator(
#     load_from_tfrecords_dir=tfrecords_dir_test,
#     quantize=False,
# )

test_generator = load_tfrecords(
    tfrecords_dir_test=tfrecords_dir_test,
    noise=noise2,                          # -1: no noise
    quantize=False,
    shuffle=True,
    digitize=True,                         # hard-digitize inputs using Part 1 thresholds
    digitize_levels=initial_levels,        # 2-bit output levels [0, 1, 2, 3]
    digitize_thresholds=thresholds,        # thresholds learned in Part 1
    seed=seed,
    test_only=True,                        # only load test generator, no train/val
)

In [ ]:
if not skip_part_2:  # fingerprint2 only exists when Part 2 training ran this session
    # Save test generator inputs and labels as numpy arrays
    # Filenames are derived from model_type and fingerprint2 to be unique per run
    # ROI-aware: dataset_test_dir reflects the active test dataset (dataset_2sc or dataset_2su_roi)
    save_data_as_npy(
        test_generator,
        x_test_npy=f"npy/{timeslices}t-{model_type}-{fingerprint2}-X_test.npy",
        y_test_npy=f"npy/{timeslices}t-{model_type}-{fingerprint2}-y_test.npy",
    )

In [ ]:
if not skip_part_2:  # model2_best only exists when Part 2 training ran this session
    # Run prediction with the best model on the test generator
    p_test = model2_best.predict(test_generator)

### MSE Evaluation

Computes Mean Squared Error between model predictions and ground truth labels on the test dataset.

In [ ]:
if not skip_part_2:  # p_test only exists when Part 2 training ran this session
    # Output labels per model type
    output_labels = {
        'Max':  ['x', 'y', 'cotA', 'cotB'],
        'Full': ['x', 'y', 'cotA', 'cotB'],
        'Slim': ['x', 'y', 'cotB'],
    }
    key = 'Max' if 'Max' in model_type else 'Full' if 'Full' in model_type else 'Slim'
    labels = output_labels[key]

    # Extract ground truth from test generator
    truth = np.concatenate([y for _, y in test_generator], axis=0)

    # MSE per output and overall
    mse_per_output = np.mean((p_test[:, :len(labels)] - truth[:, :len(labels)])**2, axis=0)
    mse_total = float(np.mean(mse_per_output))

    print(dataset_test_dir)
    print("MSE per output:")
    for label, mse in zip(labels, mse_per_output):
        print(f"  {label:>6}: {mse:.6f}")
    print(f"  {'total':>6}: {mse_total:.6f}")